In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/zalando-research/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/datasets/organizations/zalando-research/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv
/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv
/kaggle/input/datasets/organizations/zalando-research/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/datasets/organizations/zalando-research/fashionmnist/train-images-idx3-ubyte


In [2]:
import pandas as pd
import numpy as np

In [3]:
import gdown

# File ID from Google Drive share link
file_id = "1X4Hcj72NK7J2JYvgjICFj0R1XwUq1w0a"

# Construct the download URL
download_url = f"https://drive.google.com/uc?id={file_id}"

# Path to save the CSV file in Kaggle environment
output_path = "my_file.csv"

# Download the file
gdown.download(download_url, output_path, quiet=False)

# Load CSV into pandas DataFrame
import pandas as pd
df = pd.read_csv(output_path)

Downloading...
From: https://drive.google.com/uc?id=1X4Hcj72NK7J2JYvgjICFj0R1XwUq1w0a
To: /kaggle/working/my_file.csv
100%|██████████| 4.28k/4.28k [00:00<00:00, 3.11MB/s]


In [4]:
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [5]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace("?","")
    text = text.replace("''","")
    return text.split()

In [6]:
# vocab
vocab = {'<UNK>':0}

In [7]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)


In [8]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [9]:
len(vocab)

326

In [10]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 "'to": 12,
 'kill': 13,
 'a': 14,
 "mockingbird'": 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 "'1984'": 67,
 'george-orwell': 68,
 'currency': 69,
 '

In [11]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

In [13]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [14]:
dataset = QADataset(df, vocab)

In [15]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [16]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([319])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([91])
tensor([[ 10, 140,   3, 141, 272,  93, 273,   5,   3, 274]]) tensor([275])
tensor([[  1,   2,   3, 181, 182, 183, 184]]) tensor([185])
tensor([[  1,   2,   3,   4,   5, 281]]) tensor([282])
tensor([[  1,   2,   3, 222,   5, 223, 224, 225]]) tensor([226])
tensor([[ 10, 310,   3, 311, 312]]) tensor([313])
tensor([[ 42, 257,   2, 258,  83, 259, 260]]) tensor([261])
tensor([[ 10, 140,   3, 141, 142, 143, 144,  83,   3, 145]]) tensor([146])
tensor([[ 42, 137,   2, 138,  39, 139]]) tensor([53])
tensor([[ 10,  75, 111]]) tensor([112])
tensor([[ 42, 252, 253, 118, 254, 255]]) tensor([256])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([134])
tensor([[ 42, 217, 118, 218, 219,  19,  14, 220,  43]]) tensor([221])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([106])
tensor([[  1,  87, 230, 231, 232, 233]]) tensor([234])
tensor

In [17]:
import torch.nn as nn

In [18]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [19]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [20]:
learning_rate = 0.001
epochs = 20

In [21]:
model = SimpleRNN(len(vocab))


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [23]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 526.448318
Epoch: 2, Loss: 461.981858
Epoch: 3, Loss: 385.924044
Epoch: 4, Loss: 322.178091
Epoch: 5, Loss: 270.422606
Epoch: 6, Loss: 221.456407
Epoch: 7, Loss: 177.482607
Epoch: 8, Loss: 138.080103
Epoch: 9, Loss: 105.638626
Epoch: 10, Loss: 80.141861
Epoch: 11, Loss: 61.712656
Epoch: 12, Loss: 48.117138
Epoch: 13, Loss: 38.472429
Epoch: 14, Loss: 31.042550
Epoch: 15, Loss: 25.545249
Epoch: 16, Loss: 21.270651
Epoch: 17, Loss: 18.069203
Epoch: 18, Loss: 15.348273
Epoch: 19, Loss: 13.380774
Epoch: 20, Loss: 11.737145


In [24]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logics to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [25]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [26]:
list(vocab.keys())[7]

'paris'